# Yanolja 리뷰 크롤링 및 분석

이번 노트북에서는 Selenium을 사용하여 Yanolja의 호텔 리뷰 페이지에서 데이터를 크롤링하고, 수집한 데이터에 대해 분석을 진행합니다. 이 과정에서는 웹페이지 로드, 데이터 추출, 텍스트 처리 및 분석 결과를 Excel 파일로 저장하는 작업을 포함합니다.

### 1단계: Selenium으로 웹페이지 로드

Selenium을 사용하여 Yanolja 리뷰 페이지를 로드하고, 스크롤을 내려서 더 많은 데이터를 가져옵니다.

In [2]:
!pip install selenium
!pip install bs4
!pip install pandas
!pip install openpyxl

  Using cached trio-0.29.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 9.6 MB/s eta 0:00:00a 0:00:01
Using cached trio-0.29.0-py3-none-any.whl (492 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached wsproto-1.2.0-py3-none-any.whl (24 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)

[notice] A new release of pip is available:

In [3]:
from selenium import webdriver
import time

# Selenium 드라이버 설정 (Chrome 사용)
driver = webdriver.Chrome()

# Yanolja 리뷰 페이지로 이동
url = 'https://www.yanolja.com/reviews/domestic/3015391'
######## your code here ########
driver.get(url)

# 페이지 로딩을 위해 대기
time.sleep(3)

# 스크롤 설정: 페이지 하단까지 스크롤을 내리기
scroll_count = 10  # 스크롤 횟수 설정
for _ in range(scroll_count):
    ######## your code here ########
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)  # 스크롤 이후 대기

### 2단계: 페이지 소스 가져오기
웹페이지의 HTML 소스를 가져와서 BeautifulSoup을 사용해 데이터를 파싱합니다.

In [4]:
from bs4 import BeautifulSoup

# 웹페이지 소스 가져오기
page_source = driver.page_source

# BeautifulSoup를 사용하여 HTML 파싱
soup = BeautifulSoup(page_source, 'html.parser')

### 3단계: 리뷰 텍스트 추출
리뷰 텍스트를 추출하고 불필요한 공백이나 줄 바꿈을 제거합니다.

In [5]:
# 리뷰 텍스트 추출
################################
reviews_class = soup.find_all("p", class_=["context-text", "css-c92dc4"])  # `class_` 속성 사용
################################
# 심심해서 리뷰어도 추출
reviews = []
reviewers_class = []
reviewer_parents = soup.find_all("p", class_=["css-1irbwe1"])
for parent in reviewer_parents:
    span = parent.find("span")  # <p> 내부의 <span> 찾기
    if span:
        reviewers_class.append(span.text)

# 리뷰어 리스트 출력
# 각 리뷰 텍스트 정리 후 추가
for review, reviewer in zip(reviews_class, reviewers_class) :
    cleaned_text = review.get_text(strip=True).replace('\r', '').replace('\n', '')
    reviews.append(f"{reviewer} : {cleaned_text}")
print(reviews)

['로테르담궁*** : 항상 노보텔 스위트로 투숙했었는데 원하는 날짜에 스위트룸은 만실이라 일반 노보텔 룸으로 예약하여 투숙했어요서비스와 룸 컨디션 그리고 라운지 모두 훌륭하였고 재방문 의사 있어요!', '감자로* : 노보텔 앰배서더 서울 용산은 처음 방문해봤는데 이비스, 드래곤시티, 노보텔 등 여러 계열 호텔들이 붙어있어 로비에 사람이 많이 붐벼 정신없기는 했으나 호텔 내 F&B 및 편의시설을 다양하게 경험할 수 있었고, 특히 야놀자 라이브 특가 시 이벤트 당첨까지 돼 F&B 이용권까지 받아 더리본에서 런치 코스까지 즐겼습니다 ☺️✨[장점] - 널찍한 주차 공간 - F&B 및 부대 시설(헬스, 수영장) 구비 - 깔끔하니 쾌적한 객실 - 객실 내 전자레인지 구비 [아쉬운 점] - 다소 올드한 객실 인테리어 - 전망 (창밖으로 공사장뷰가 보임)', '시애틀달리*** : 청결을 매우 중요시하는 여친이 좋아했고요욕실과 화장실 분리되있어서좋았어요방이큰편이고 아주편안한 소파가있는데 햇살이 너무잘들어와서 여친이 사진잘나온다고 거기서 사진찍느라 정신없더라고여추석때라 체크인 오래걸릴까봐 살짝걱정했는데 전혀오래안걸렸고요 직원분들 너무친절하고 물필요하면 바로가져다주시고 좋았어요 근데 얼음이없는건 조금아쉽네요용산역에서 3번출구 나오면 에스컬레이터나 엘레베이터로바로 갈수있어서 좋았고용산 아이파크도 가까워서 접근성 최고에요추석 이곳에서 즐겁게잘보냈고 다음에 또 방문하겠습니다☺️', '우면산작가** : 연말 평일에 방문했습니다체크인 시간은 딱 맞춰갔는데 대기는 따로 없었고 호텔 사이트 가입하니까 23층으로 주셔서 좋았어요방 컨디션에 대한 나쁜 후기가 좀 있어서 걱정했는데 저는 룸컨디션 너무 좋았습니다 침대도 푹신하고냉장고가 너무 작아서 케이크가 안 들어간게 좀 아쉬웠지만 그거 제외하고는 다 괜찮았네요수영장은 늦은 시간에 가면 생각보다 사람 얼마 없이 즐길 수 있습니다전반적으로 가성비 좋은 5성급 숙소로 추천드려요!', '말죽거리버**** : 용산역과 연결되어 있어 길 헤맬 일도 없고 

### 4단계: 별점 데이터 추출
HTML에서 별점 데이터를 추출하고, 각 리뷰의 별점을 계산합니다.

In [6]:
ratings = []

rating_containers = soup.find_all("svg", attrs={"xmlns": "http://www.w3.org/2000/svg", "class": "css-1mj121y"})

count = 0
rating = 0

for container in rating_containers:
    if count == 5:
        ratings.append(rating)
        count = 0
        rating = 0

    path_tag = container.find("path")
    if path_tag:
        d_attr = path_tag['d']
        star_check = d_attr[1:3] 

        # 너의 기준에 맞게 판단
        if star_check.startswith('12'):  # 별이 채워진 경우로 간주
            rating += 1

    count += 1

# 마지막 리뷰 처리 (남은 게 있을 경우)
if count > 0:
    ratings.append(rating)

print(ratings)

[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 3, 5, 5, 4, 5, 4, 4, 5, 5, 5, 4, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 5, 3, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 5, 5, 4, 3, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]


### 5단계: 데이터 정리 및 DataFrame으로 변환
수집된 데이터를 Pandas DataFrame으로 변환하여 후속 분석을 용이하게 만듭니다.

In [7]:
import pandas as pd

# 별점과 리뷰를 결합하여 리스트 생성
data = list(zip(ratings, reviews))

# DataFrame으로 변환
df_reviews = pd.DataFrame(data, columns=['Rating', 'Review'])
df_reviews

,Rating,Review
0,5,로테르담궁*** : 항상 노보텔 스위트로 투숙했었는데 원하는 날짜에 스위트룸은 만실...
1,5,"감자로* : 노보텔 앰배서더 서울 용산은 처음 방문해봤는데 이비스, 드래곤시티, 노..."
2,5,시애틀달리*** : 청결을 매우 중요시하는 여친이 좋아했고요욕실과 화장실 분리되있어...
3,5,우면산작가** : 연말 평일에 방문했습니다체크인 시간은 딱 맞춰갔는데 대기는 따로 ...
4,5,말죽거리버**** : 용산역과 연결되어 있어 길 헤맬 일도 없고 편합니다. 주변에 ...
...,...,...
215,5,예쁜우******* : 말 할 필요없이 최고의 시설이였어용다만 이벤트덕분에 사람이 ...
216,5,연* : 숙소 너무 깔끔했고 남산타워 뷰까지 최고였어요! 편의점도 2층에 있어서 멀...
217,5,히히히****** : 서울에서 주차되는 숙박 이가격에 찾기 힘든데 너무 좋았어요!!...
218,5,비앙새* : 너무나 좋고 조식도 조금 차가운거 빼곤 맛있었어요 재방문이였지만 위치도...


### 6단계: 리뷰 분석 - 평균 별점 계산
수집된 리뷰에서 평균 별점을 계산합니다.

In [8]:
# 평균 별점 계산
average_rating = sum(ratings) / len(ratings)
print(f"{average_rating:.2f}")

4.81


### 7단계: 자주 등장하는 단어 추출
리뷰 텍스트에서 자주 등장하는 단어를 추출하고, 불용어를 제거하여 분석합니다.

In [9]:
from collections import Counter
import re

# 불용어 리스트 (한국어)
korean_stopwords = set(['이', '그', '저', '것', '들', '다', '을', '를', '에', '의', '가', '이', '는', '해', '한', '하', '하고', '에서', '에게', '과', '와', '너무', '잘', '또','좀', '호텔', '아주', '진짜', '정말'])

# 모든 리뷰를 하나의 문자열로 결합
all_reviews_text = " ".join(reviews)

# 단어 추출 (특수문자 제거)
words = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', all_reviews_text).split()

# 불용어 제거
filtered_words = [c for c in words if c not in korean_stopwords]

# 단어 빈도 계산
word_counts = Counter(filtered_words)

# 자주 등장하는 상위 15개 단어 추출
common_words = word_counts.most_common(15)
print(common_words)

[('좋았어요', 42), ('있어서', 38), ('좋았습니다', 35), ('좋고', 32), ('바로', 28), ('수', 22), ('좋은', 21), ('깔끔하고', 20), ('깨끗하고', 20), ('노보텔', 17), ('체크인', 17), ('객실', 16), ('그래도', 16), ('조금', 16), ('수영장', 15)]


### 8단계: 분석 결과 요약
평균 별점과 자주 등장하는 단어를 DataFrame으로 만들어 최종 분석 결과를 요약합니다.

In [10]:
# 분석 결과 요약
summary_df = pd.DataFrame({
    'Average Rating': [average_rating],
    'Common Words': [', '.join([f"{word}({count})" for word, count in common_words])]
})

# 최종 DataFrame 결합
final_df = pd.concat([df_reviews, summary_df], ignore_index=True)
final_df

,Rating,Review,Average Rating,Common Words
0,5.0,로테르담궁*** : 항상 노보텔 스위트로 투숙했었는데 원하는 날짜에 스위트룸은 만실...,NaN,NaN
1,5.0,"감자로* : 노보텔 앰배서더 서울 용산은 처음 방문해봤는데 이비스, 드래곤시티, 노...",NaN,NaN
2,5.0,시애틀달리*** : 청결을 매우 중요시하는 여친이 좋아했고요욕실과 화장실 분리되있어...,NaN,NaN
3,5.0,우면산작가** : 연말 평일에 방문했습니다체크인 시간은 딱 맞춰갔는데 대기는 따로 ...,NaN,NaN
4,5.0,말죽거리버**** : 용산역과 연결되어 있어 길 헤맬 일도 없고 편합니다. 주변에 ...,NaN,NaN
...,...,...,...,...
216,5.0,연* : 숙소 너무 깔끔했고 남산타워 뷰까지 최고였어요! 편의점도 2층에 있어서 멀...,NaN,NaN
217,5.0,히히히****** : 서울에서 주차되는 숙박 이가격에 찾기 힘든데 너무 좋았어요!!...,NaN,NaN
218,5.0,비앙새* : 너무나 좋고 조식도 조금 차가운거 빼곤 맛있었어요 재방문이였지만 위치도...,NaN,NaN
219,5.0,고냥이******* : 용산역과 연계성이 좋고 뷰와 룸컨디션이 훌륭했습니다. 칫솔 ...,NaN,NaN


### 9단계: Excel 파일로 저장
최종 결과를 Excel 파일로 저장합니다.

In [11]:
# Excel 파일로 저장
final_df.to_excel("final.xlsx", index = False)

### 10단계: 드라이버 종료
크롤링이 끝난 후, Selenium 드라이버를 종료합니다.

In [12]:
# 드라이버 종료
driver.quit()